# **RAG (Retrieval-Augmented Generation)**
This is a RAG pipeline designed for specialized legal documents.

**Phase 1: Setup and Ingestion**

This part prepares my environment and converts a static PDF into a format the AI can "search."

In [2]:
# Install RAG dependencies
# I installed langchained for the workflow, sentence-transformers for text-to-vector math,
# chromadb for storage, and transformers/accelerate to run the actual AI model.
!pip install langchain langchain-community sentence-transformers chromadb pypdf bitsandbytes accelerate

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM

# Ingest the PDF
# PyPDFLoader reads the PDF and splits it into individual pages with metadata.
loader = PyPDFLoader("tax_law.pdf")
pages = loader.load()

# Chunking strategy
# AI models have "context windows" (limit on how much text they can read at once).
# Therefore, I split long pages into smaller 500-character chunks.
# The 'overlap' ensures that a sentence cut in half in one chunk is finished in the next.
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 150)
chunks = text_splitter.split_documents(pages)

**Phase 2: Vectorization and Model Loading**

Here, I turned human language into numbers (embeddings) so the computer could find "similar" meanings.

In [ ]:
from huggingface_hub import login

# Paste your token inside the quotes
login("hugging_face_token")

# Create local embeddings (German-optimized)
# This model takes a text chunk and turns it into a numerical vector.
# I used a 'multilingual' model because it understands German legal nuances better.
embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Initialize Vector Database
# Chroma takes the chunks and their numerical vectors and saves them to a folder.
# This allows me to search thousands of pages in milliseconds later.
vector_db = Chroma.from_documents(chunks, embeddings, persist_directory = "./tax_rag_db")
print(f"RAG DB initialized with {len(chunks)} searchable tax segments.")

# Initialize the tokenizer and LLM
# I used BLOOMZ, which is great for following instructions in multiple languages.
model_name_llm = "google/gemma-1.1-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name_llm)

# Load model without 8-bit quantization
# Note: bitsandbytes/quantization is temporarily skipped here for compatibility.
model = AutoModelForCausalLM.from_pretrained(
    model_name_llm,
    device_map = "auto" # This automatically finds the GPU
)

/tmp/ipykernel_824/3564851697.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

**Phase 3: The RAG Logic**

This function is the "brain" of the operation. It finds the law and hands it to the AI.

In [4]:
import pandas as pd
import csv
from tqdm import tqdm

def ask_tax_rag(question):
    # 1. Retrieve the top 3 relevant law passages/chunks in our DB most similar to the user's question.
    docs = vector_db.similarity_search(question, k = 3)

    # Clean up the retrieved chunks and attach their PDF page numbers.
    context = ""
    for d in docs:
        page_num = d.metadata.get('page', 'unknown')
        context += f"[Seite {page_num}]: {d.page_content}\n"

    # 2. Create a specific (RAG) prompt.
    # We "pin" the model's personality as a tax expert and give it the specific context found above.
    rag_prompt = f"""<start_of_turn>user
Du bist ein Experte für das österreichische Steuerrecht. Beantworte die Frage kurz und präzise basierend auf dem KONTEXT.
Nenne immer das Gesetz und die Seitenzahl.

KONTEXT:
{context}

FRAGE:
{question}<end_of_turn>
<start_of_turn>model
"""

    # 3. Generate the answer using the FINE-TUNED model
    # Convert text to tokens and let the model generate the answer, and record prompt length
    # Ensure inputs are on the correct device (model's device)
    inputs = tokenizer(rag_prompt, return_tensors = "pt").to(model.device)
    input_length = inputs.input_ids.shape[1]

    # Generate the answer
    outputs = model.generate(
        **inputs,
        max_new_tokens = 150, # Lowered to 150 for speed
        do_sample = False,
        eos_token_id = tokenizer.eos_token_id,
    )

    # Decode the numerical output back into German text.
    full_response = tokenizer.decode(outputs[0], skip_special_tokens = True)

    # TOKEN SLICING: This removes the prompt so your CSV only has the answer
    generated_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer.strip()

**Phase 4: Batch Processing**

This final part automated the process for my entire dataset.

In [5]:
# 4. Test with only the first 3 rows
test_df = pd.read_csv('dataset_clean.csv')
test_sample = test_df.head(3) # Take only the first 3 rows
rag_results = []

print("Starting test run for 3 rows...")

for _, row in test_sample.iterrows():
    answer = ask_tax_rag(row['prompt'])
    print(f"\nID: {row['id']}")
    print(f"Answer: {answer}") # Check if this looks like a real sentence now!
    rag_results.append({"id": row['id'], "answer": answer})

# Preview the temporary results dataframe
output_df = pd.DataFrame(rag_results)
print("\nPreview")
print(output_df)

Starting test run for 3 rows...

ID: CORP-TAX-001
Answer: Die textgegebene Information ist nicht relevant für die Bestimmung der Körperschaftsteuer und kann daher nicht beantwortet werden.

ID: CORP-TAX-002
Answer: Es gibt keine Steuerbegrenzungen oder Regelungen, die die Behandlung von zinslosen Darlehen an Gesellschafter regeln.

ID: CORP-TAX-003
Answer: Die Körperschaften, die sämtliche Einkünfte den Einkünften aus Gewerbebetrieb zuzurechnen sind, sind Unternehmen mit mehr als 25 % Beteiligung an Gesell ­schafter­ Geschäftsführer/innen.

Preview
             id                                             answer
0  CORP-TAX-001  Die textgegebene Information ist nicht relevan...
1  CORP-TAX-002  Es gibt keine Steuerbegrenzungen oder Regelung...
2  CORP-TAX-003  Die Körperschaften, die sämtliche Einkünfte de...


In [6]:
# 4. Process the csv dataset containing 644 tax questions
test_df = pd.read_csv('dataset_clean.csv')

rag_results = []

# Loop through every question in the CSV.
# 'tqdm' provides a progress bar
for _, row in tqdm(test_df.iterrows(), total = len(test_df)):
    answer = ask_tax_rag(row['prompt'])

    # Store the ID and the AI's answer together.
    rag_results.append({"id": row['id'], "answer": answer})

# 5. Export final results
# Save everything to a new CSV for submission or review.
output_df = pd.DataFrame(rag_results)
output_df.to_csv("rag_results.csv", index = False, quoting = csv.QUOTE_ALL)
print("Task 3 Complete! File saved as rag_results.csv")

100%|██████████| 643/643 [22:57<00:00,  2.14s/it]

Task 3 Complete! File saved as rag_results.csv
